# 04. Weather and the information cut-off

**Where we are:** notebook 03 showed that weather drives most of what the calendar cannot:
temperature through heating, and sunshine through embedded solar, which removes about 1 MW of
`ND` for every MW it generates.

**The trap:** the weather that actually happened is not known a day ahead. A model trained and
tested on actual weather would look far better than any real forecast could be. NESO had only
weather forecasts when it published, so our model must be tested on weather forecasts too.

**What this notebook does:**

1. Fixes exactly what our forecast is allowed to know, and when.
2. Checks a live source of demand, so the lag features are honest.
3. Downloads actual weather (for training) and archived day-ahead forecasts (for testing).
4. Measures how far the forecasts are from what happened, and what that costs in MW.

**Background reading**
- [Open-Meteo Previous Runs API](https://open-meteo.com/en/docs/previous-runs-api): archived forecasts at fixed lead times
- [Open-Meteo Historical Weather API](https://open-meteo.com/en/docs/historical-weather-api): ERA5 reanalysis
- [Forecasting: Principles and Practice, section 7.6, forecasting with regression](https://otexts.com/fpp3/forecasting-regression.html): why predictors must themselves be known in advance


## Step 1. Set up


In [1]:
import time
from pathlib import Path

import holidays
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests

RAW = Path("../data/raw")
PROCESSED = Path("../data/processed")
TEST_START = pd.Timestamp("2024-06-01", tz="Europe/London")

BLUE, ORANGE = "#2a78d6", "#eb6834"
plt.rcParams.update({
    "figure.dpi": 90,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.color": "#e4e4e0",
    "lines.linewidth": 2,
})

demand = pd.read_parquet(PROCESSED / "demand.parquet")


def get_json(url, params, tries=5):
    """GET with a pause and retry when the API says too many requests (HTTP 429)."""
    for _ in range(tries):
        response = requests.get(url, params=params, timeout=300)
        if response.status_code != 429:
            response.raise_for_status()
            return response.json()
        time.sleep(65)
    response.raise_for_status()


## Step 2. The forecast origin

**What:** state, as code, the moment our forecast is made.

**Why:** every feature has to pass one test: was it known at that moment? Writing the origin
down once, as a function, means every later notebook uses the same rule.

Notebook 02 found NESO publishes between 08:33 and 10:33 UTC on the day before. Our origin is
**08:30 UTC on the day before the target day**, which is never later than NESO.

The target day runs from local midnight to local midnight, so from the origin the forecast
covers lead times of roughly 15 to 40 hours:


In [2]:
ORIGIN_TIME = pd.Timedelta(hours=8, minutes=30)


def forecast_origin(target_date):
    """UTC moment the forecast for a local target date is made: 08:30 UTC the day before."""
    day_before = pd.Timestamp(target_date) - pd.Timedelta(days=1)
    return (day_before + ORIGIN_TIME).tz_localize("UTC")


for target in ["2025-01-15", "2025-07-15"]:
    origin = forecast_origin(target)
    first = pd.Timestamp(target).tz_localize("Europe/London").tz_convert("UTC")
    last_end = (pd.Timestamp(target) + pd.Timedelta(days=1)).tz_localize("Europe/London").tz_convert("UTC")
    print(f"target {target}: origin {origin:%Y-%m-%d %H:%M} UTC, "
          f"lead {(first - origin) / pd.Timedelta(hours=1):.1f} to {(last_end - origin) / pd.Timedelta(hours=1):.1f} hours")


target 2025-01-15: origin 2025-01-14 08:30 UTC, lead 15.5 to 39.5 hours
target 2025-07-15: origin 2025-07-14 08:30 UTC, lead 14.5 to 38.5 hours


## Step 3. Is recent demand known at the origin?

**What:** download Elexon's *initial* national demand outturn (`INDO`) and check when it is
published and how close it is to `ND`.

**Why:** NESO's historic demand files are published weeks after the fact, so they cannot be a
live input. Notebook 03 planned to use only demand from two or more days back for that reason.
But GB demand is also published within minutes by Elexon, the market's settlement body. If that
live figure matches `ND`, the forecast can also use demand from the morning of the day before.

**Why the same source for training:** `INDO` is an early estimate that NESO later revises into
`ND`. If the model learns from revised `ND` lags but is fed early `INDO` lags in use, the inputs
would differ between training and testing. Building lag features from `INDO` in both removes
that mismatch.

**Further reading:** [Elexon Insights API](https://developer.data.elexon.co.uk), dataset
`INDO` (initial national demand outturn)


In [3]:
INDO_FILE = RAW / "elexon_indo.parquet"
ELEXON = "https://data.elexon.co.uk/bmrs/api/v1/demand/outturn"

if not INDO_FILE.exists():
    chunks = []
    start, end = pd.Timestamp("2018-01-01"), pd.Timestamp.now().normalize() - pd.Timedelta(days=1)
    while start <= end:
        stop = min(start + pd.Timedelta(days=27), end)  # the API allows 28 days per request
        response = requests.get(ELEXON, params={
            "settlementDateFrom": f"{start:%Y-%m-%d}", "settlementDateTo": f"{stop:%Y-%m-%d}", "format": "json",
        }, timeout=60)
        response.raise_for_status()
        chunks.append(pd.DataFrame(response.json()["data"]))
        start = stop + pd.Timedelta(days=1)
        time.sleep(0.2)
    pd.concat(chunks, ignore_index=True).to_parquet(INDO_FILE, index=False)

indo = pd.read_parquet(INDO_FILE)
indo["start_utc"] = pd.to_datetime(indo["startTime"], utc=True)
indo["publish_utc"] = pd.to_datetime(indo["publishTime"], utc=True)
indo = indo.rename(columns={"settlementDate": "Date", "settlementPeriod": "Settlement_Period",
                            "initialDemandOutturn": "INDO"})
print(indo.shape, indo["Date"].min(), "to", indo["Date"].max())
print("repeated half-hours:", int(indo.duplicated("start_utc").sum()))

delay = (indo["publish_utc"] - (indo["start_utc"] + pd.Timedelta(minutes=30))).dt.total_seconds() / 60
print("minutes from end of period to publication:")
print(delay.describe().round(1).to_string())
print(f"published more than 30 minutes after the period ends: {(delay > 30).mean():.4%}")


(154722, 8) 2018-01-01 to 2026-11-11
repeated half-hours: 0
minutes from end of period to publication:
count    154722.0
mean          0.2
std           5.9
min           0.0
25%           0.0
50%           0.0
75%           0.0
max        1193.0
published more than 30 minutes after the period ends: 0.1713%


In [4]:
indo = indo.drop_duplicates("start_utc", keep="last")
both = demand[["start_utc", "ND"]].merge(indo[["start_utc", "INDO"]], on="start_utc", how="inner")
both["gap"] = both["INDO"] - both["ND"]
both["year"] = both["start_utc"].dt.year

summary = both.groupby("year").agg(
    rows=("gap", "size"),
    mean_gap=("gap", "mean"),
    mean_abs_gap=("gap", lambda g: g.abs().mean()),
    corr=("INDO", lambda s: s.corr(both.loc[s.index, "ND"])),
)
summary.round(3)


,rows,mean_gap,mean_abs_gap,corr
year,,,,
2018,17376,-62.695,75.627,1.0
2019,17424,-51.311,70.118,1.0
2020,17398,-41.323,62.145,1.0
2021,17280,-36.811,53.477,1.0
2022,17518,-35.049,55.863,1.0
2023,17518,-36.413,59.034,1.0
2024,17568,-35.585,58.040,1.0
2025,17520,-36.963,59.364,1.0
2026,14014,-25.925,47.876,1.0


**Found:**

- `INDO` is published as each half-hour ends: the median delay is zero minutes. Only 0.17% of
  periods came out more than 30 minutes late (the longest delay was about 20 hours). So at the origin, demand is
  known up to the morning of the day before, not only up to two days before.
- It tracks `ND` closely: the correlation rounds to 1.00, and the mean absolute gap is 48 to 76 MW,
  about 0.2% of demand. `INDO` runs about 36 MW below `ND` in recent years (63 MW in 2018), which
  is the size of the revisions NESO makes later.

**Decision:**
- Lag features come from `INDO`, in training and in testing, so the inputs mean the same thing
  in both. The target stays `ND`.
- The forecast may use `INDO` up to 08:00 UTC on the day before (Step 11 explains the margin).
  This replaces the plan's "two days or more" rule, which was stricter than needed. Notebook 03
  found yesterday's demand (correlation 0.915) only a little more informative than last week's
  (0.911), so Day 6 will check how much the extra morning of data helps.
